<a href="https://colab.research.google.com/github/igorcsouza/marca-dagua/blob/main/marca_dagua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Instruções

Como usar

1. Faça uma cópia deste Colab para o seu Google Drive
2. Criar no próprio Drive a pasta marcaDagua e dentro dela, resource e gerados
3. Colocar o roteiro e a planilha na pasta resource
4. Configurar o ID_PLANILHA na célula "Configuração", no Colab
5. Na planilha, preencher:
  *   Nome do arquivo PDF do roteiro a ser enviado
  *   Versão do roteiro
  *   Assunto do e-mail
  *   Corpo do e-mail
  *   Lista de destinatários
6. Executar as 3 células no Colab
7. Os PDFs personalizados aparecem na pasta gerados
8. Depois, no Apps Script da própria planilha, executar enviarTodos() para fazer os envios.

# 1. Configuração inicial

In [ ]:
# ==========================================
# CONFIGURAÇÃO
# ==========================================

ID_PLANILHA = "1m7aDVfkdiqunbmrYyhNkkM80ccHuNv5Jsona5V4vV9U"

# ==========================================

NOME_ABA = "Página1"

NOME_PASTA_PROJETO = "marcaDagua"
NOME_PASTA_RECURSOS = "resource"
NOME_PASTA_GERADOS = "gerados"

# ==========================================
# INSTALAR PACOTES
# ==========================================

!pip install pypdf reportlab -q

print("Pacotes instalados!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.5 MB/s eta 0:00:00
Pacotes instalados!


# 2. Lê planilha

In [ ]:
# ==========================================
# CONEXÃO COM GOOGLE SHEETS E DRIVE
# ==========================================

from google.colab import auth, drive
from googleapiclient.discovery import build

import os
import io
import pandas as pd

# Autenticação
auth.authenticate_user()

# Google Sheets
sheets = build("sheets", "v4")

# Google Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

print("Conexão com Google concluída!")

# ==========================================
# ENCONTRAR PASTAS
# ==========================================

import os

CAMINHO_PROJETO = os.path.join(
    "/content/drive/My Drive",
    NOME_PASTA_PROJETO
)

CAMINHO_RECURSOS = os.path.join(
    CAMINHO_PROJETO,
    NOME_PASTA_RECURSOS
)

CAMINHO_GERADOS = os.path.join(
    CAMINHO_PROJETO,
    NOME_PASTA_GERADOS
)


# ==========================================
# VERIFICAR PASTAS
# ==========================================

if not os.path.isdir(CAMINHO_PROJETO):
    raise FileNotFoundError(
        f"Pasta do projeto não encontrada:\n{CAMINHO_PROJETO}"
    )

if not os.path.isdir(CAMINHO_RECURSOS):
    raise FileNotFoundError(
        f"Pasta resource não encontrada:\n{CAMINHO_RECURSOS}"
    )

if not os.path.isdir(CAMINHO_GERADOS):
    raise FileNotFoundError(
        f"Pasta gerados não encontrada:\n{CAMINHO_GERADOS}"
    )


print("Pastas encontradas com sucesso!")
print("Projeto:  ", CAMINHO_PROJETO)
print("Recursos: ", CAMINHO_RECURSOS)
print("Gerados:  ", CAMINHO_GERADOS)

# ==========================================
# 5. CONFIGURAÇÃO E LEITURA DA PLANILHA
# ==========================================

ID_PLANILHA = "1m7aDVfkdiqunbmrYyhNkkM80ccHuNv5Jsona5V4vV9U"
NOME_ABA = "Página1"

COLUNA_NOME = "Nome"
COLUNA_EMAIL = "E-mail"
COLUNA_ENVIA = "Envia?"

# ==========================================
# LER GOOGLE SHEETS
# ==========================================

resultado = sheets.spreadsheets().values().get(
    spreadsheetId=ID_PLANILHA,
    range=f"{NOME_ABA}!A:Z"
).execute()

valores = resultado.get("values", [])

if not valores:
    raise ValueError("A planilha não retornou nenhum dado.")


# ==========================================
# ENCONTRAR O CABEÇALHO
# ==========================================

linha_cabecalho = None

for numero_linha, linha in enumerate(valores):

    if (
        COLUNA_NOME in linha and
        COLUNA_EMAIL in linha and
        COLUNA_ENVIA in linha
    ):
        linha_cabecalho = numero_linha
        break


if linha_cabecalho is None:
    raise ValueError(
        "Não encontrei as colunas necessárias: "
        + ", ".join([
            COLUNA_NOME,
            COLUNA_EMAIL,
            COLUNA_ENVIA
        ])
    )


cabecalho = valores[linha_cabecalho]

print(
    f"Cabeçalho encontrado na linha {linha_cabecalho + 1}:"
)
print(cabecalho)


# ==========================================
# TRANSFORMAR EM DATAFRAME
# ==========================================

linhas = valores[linha_cabecalho + 1:]

# Garantir que todas as linhas tenham o mesmo número
# de colunas do cabeçalho
linhas_normalizadas = []

for linha in linhas:

    linha = linha + [""] * (len(cabecalho) - len(linha))

    linhas_normalizadas.append(
        linha[:len(cabecalho)]
    )


df = pd.DataFrame(
    linhas_normalizadas,
    columns=cabecalho
)


# ==========================================
# FILTRAR QUEM DEVE RECEBER
# ==========================================

df[COLUNA_ENVIA] = (
    df[COLUNA_ENVIA]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

df_gerar = df[
    (df[COLUNA_ENVIA] == "S")
].copy()


# ==========================================
# RESULTADO
# ==========================================

print()
print(f"Total de pessoas na tabela: {len(df)}")
print(f"PDFs a gerar: {len(df_gerar)}")

display(
    df_gerar[
        [
            COLUNA_NOME,
            COLUNA_EMAIL,
            COLUNA_ENVIA
        ]
    ]
)

# ==========================================
# 6. CONFIGURAÇÃO DO PDF
# ==========================================

# ------------------------------------------
# Ler "Arquivo PDF" da área de configuração
# ------------------------------------------

ARQUIVO_PDF_NOME = None

for linha in valores[:linha_cabecalho]:

    if len(linha) >= 2:
        chave = str(linha[0]).strip()
        valor = str(linha[1]).strip()

        if chave == "Arquivo PDF":
            ARQUIVO_PDF_NOME = valor
            break

if not ARQUIVO_PDF_NOME:
    raise ValueError(
        "Não encontrei 'Arquivo PDF' na área de configuração."
    )


# ------------------------------------------
# Caminhos no Google Drive
# ------------------------------------------

ARQUIVO_PDF = os.path.join(
    CAMINHO_RECURSOS,
    ARQUIVO_PDF_NOME
)

PASTA_SAIDA = CAMINHO_GERADOS


# ------------------------------------------
# Criar pasta de saída
# ------------------------------------------

os.makedirs(
    PASTA_SAIDA,
    exist_ok=True
)


# ------------------------------------------
# Verificar PDF original
# ------------------------------------------

if not os.path.exists(ARQUIVO_PDF):

    raise FileNotFoundError(
        f"PDF original não encontrado:\n{ARQUIVO_PDF}"
    )


# ------------------------------------------
# Mostrar configuração
# ------------------------------------------

print("Configuração do PDF:")
print(f"  Arquivo: {ARQUIVO_PDF_NOME}")
print(f"  Entrada: {ARQUIVO_PDF}")
print(f"  Saída:   {PASTA_SAIDA}")

Conexão com Google concluída!
Pastas encontradas com sucesso!
Projeto:   /content/drive/My Drive/marcaDagua
Recursos:  /content/drive/My Drive/marcaDagua/resource
Gerados:   /content/drive/My Drive/marcaDagua/gerados


Cabeçalho encontrado na linha 7:
['Nome', 'E-mail', 'Envia?', 'Versão enviada', 'Data envio']

Total de pessoas na tabela: 2
PDFs a gerar: 2


,Nome,E-mail,Envia?
0,Igor,igorcsouza@gmail.com,S
1,Martha Carvalho,igorcsouza@gmail.com,S


Configuração do PDF:
  Arquivo: roteiro_v2.pdf
  Entrada: /content/drive/My Drive/marcaDagua/resource/roteiro_v2.pdf
  Saída:   /content/drive/My Drive/marcaDagua/gerados


# 3. Geração de PDF

In [ ]:
# ==========================================
# GERAR PDFs COM MARCA-D'ÁGUA
# ==========================================

from pypdf import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from reportlab.lib.colors import Color


# ------------------------------------------
# Criar marca-d'água
# ------------------------------------------

def criar_marca_dagua(largura, altura, texto):

    buffer = io.BytesIO()

    c = canvas.Canvas(
        buffer,
        pagesize=(largura, altura)
    )

    # Cor + transparência
    # (Para escurecer, diminuir os valores do Color [manter o alpha])
    c.setFillColor(
        Color(
            0.65,
            0.65,
            0.65,
            alpha=0.2
        )
    )

    # Fonte
    c.setFont(
        "Helvetica-Bold",
        65
    )

    # Posicionar no centro
    c.saveState()

    c.translate(
        largura / 2,
        altura / 2
    )

    c.rotate(45)

    c.drawCentredString(
        0,
        0,
        texto
    )

    c.restoreState()

    c.save()

    buffer.seek(0)

    return buffer


# ------------------------------------------
# Adicionar marca-d'água ao PDF
# ------------------------------------------

def adicionar_marca_dagua(
    arquivo_entrada,
    arquivo_saida,
    texto
):

    reader = PdfReader(arquivo_entrada)
    writer = PdfWriter()

    for pagina in reader.pages:

        largura = float(
            pagina.mediabox.width
        )

        altura = float(
            pagina.mediabox.height
        )

        marca = criar_marca_dagua(
            largura,
            altura,
            texto
        )

        camada = PdfReader(
            marca
        ).pages[0]

        pagina.merge_page(
            camada
        )

        writer.add_page(
            pagina
        )

    with open(
        arquivo_saida,
        "wb"
    ) as arquivo:

        writer.write(arquivo)


# ------------------------------------------
# Determinar prefixo do arquivo
# ------------------------------------------

prefixo = os.path.splitext(
    ARQUIVO_PDF_NOME
)[0]


# ------------------------------------------
# Gerar PDFs
# ------------------------------------------

for _, pessoa in df_gerar.iterrows():

    nome = str(
        pessoa[COLUNA_NOME]
    ).strip()

    # Nome seguro para arquivo
    nome_arquivo = nome

    for caractere in [
        "/", "\\", ":", "*", "?",
        '"', "<", ">", "|"
    ]:
        nome_arquivo = nome_arquivo.replace(
            caractere,
            "_"
        )

    nome_arquivo = nome_arquivo.replace(
        " ",
        "_"
    )

    # Nome final
    nome_pdf = (
        prefixo
        + "_"
        + nome_arquivo
        + ".pdf"
    )

    caminho_saida = os.path.join(
        PASTA_SAIDA,
        nome_pdf
    )

    adicionar_marca_dagua(
        ARQUIVO_PDF,
        caminho_saida,
        nome
    )

    print(
        f"✓ {nome} → {nome_pdf}"
    )


print()
print("Geração concluída!")

✓ Igor → roteiro_v2_Igor.pdf
✓ Martha Carvalho → roteiro_v2_Martha_Carvalho.pdf

Geração concluída!
